
# 🚀 Advanced AML System with Transformer NLP + Risk Threshold Dashboard

This notebook includes:

- Transformer-based name embeddings (Sentence Transformers)
- Semantic similarity scoring
- ML Risk Model
- Accuracy & Confusion Matrix
- ROC & AUC
- Risk Threshold Interactive Dashboard
- Business Impact Simulation

NOTE: Requires internet on first run to download transformer model.


In [ ]:

!pip install sentence-transformers scikit-learn pandas numpy matplotlib ipywidgets joblib


In [ ]:

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display
import joblib

np.random.seed(42)


## 🔹 Load Transformer Model

In [ ]:

model_transformer = SentenceTransformer('all-MiniLM-L6-v2')


## 🔹 Generate Synthetic Name Dataset

In [ ]:

names_watchlist = ["Muhammad Ali", "Ahmed Khan", "John Smith", "Ali Hassan"]
countries = ["India","Iran","Syria","USA"]

data = []
labels = []

for _ in range(800):
    name1 = np.random.choice(names_watchlist)
    name2 = np.random.choice(names_watchlist)
    
    emb1 = model_transformer.encode(name1)
    emb2 = model_transformer.encode(name2)
    
    similarity = cosine_similarity([emb1],[emb2])[0][0]
    
    dob = np.random.choice([0,1], p=[0.7,0.3])
    country = np.random.choice([0,1], p=[0.6,0.4])
    severity = np.random.uniform(0.3,1.0)
    
    label = 1 if (similarity > 0.8 and dob==1 and country==1) else 0
    
    data.append([similarity,dob,country,severity])
    labels.append(label)

X = np.array(data)
y = np.array(labels)

print("Dataset:",X.shape)


## 🔹 Train Risk Model

In [ ]:

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

model = LogisticRegression()
model.fit(X_train,y_train)

preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:,1]

accuracy = accuracy_score(y_test,preds)
print("Accuracy:",accuracy)


## 🔹 Confusion Matrix

In [ ]:

cm = confusion_matrix(y_test,preds)

plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.show()


## 🔹 ROC Curve & AUC

In [ ]:

fpr,tpr,_ = roc_curve(y_test,probs)
roc_auc = auc(fpr,tpr)

plt.figure()
plt.plot(fpr,tpr)
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

print("AUC:",roc_auc)


## 🔹 Risk Threshold Dashboard

In [ ]:

def evaluate_threshold(threshold):
    preds_custom = (probs >= threshold).astype(int)
    cm_custom = confusion_matrix(y_test,preds_custom)
    acc = accuracy_score(y_test,preds_custom)
    
    plt.figure()
    plt.imshow(cm_custom)
    plt.title(f"Confusion Matrix @ Threshold={threshold:.2f}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.colorbar()
    plt.show()
    
    print("Accuracy:",acc)
    print("High Risk Alerts:",sum(preds_custom))

slider = widgets.FloatSlider(value=0.5,min=0.1,max=0.9,step=0.05)
widgets.interact(evaluate_threshold,threshold=slider)


## 🔹 Business Impact Simulation

In [ ]:

def business_impact(threshold):
    preds_custom = (probs >= threshold).astype(int)
    total_alerts = len(preds_custom)
    high_alerts = sum(preds_custom)
    reduction = (1 - high_alerts/total_alerts) * 100
    
    print("Total Alerts:",total_alerts)
    print("Alerts Sent to Analyst:",high_alerts)
    print("Workload Reduction %:",reduction)

business_impact(0.5)
